# CMB Summer School: Machine Learning for CMB Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flatironinstitute/2026-flatiron-cmb-summer-school/blob/main/notebooks/CMB_School_Machine_Learning.ipynb)

> **Running on Colab:** pick a GPU runtime (*Runtime → Change runtime type → GPU*),
> then *Run all*. The first cell installs the packages and downloads the dataset
> (~720 MB) and the diffusion checkpoint from this repo's GitHub Release — a few
> minutes the first time.

**Three modern ML tools, and how to trust them.**

This notebook has **one organizing idea**, and it matters more than any single
method: *a learned method is only as good as your ability to validate it against
something you already trust.* Every act pairs a learned tool with a classical foil
and holds it to a concrete, quantitative bar:

| Act | Learned method | Classical foil | Held to the bar of… |
|-----|----------------|----------------|----------------------|
| 1 | **Emulator** of the CMB spectra | CAMB | **cosmic variance** |
| 2 | **SBI** of cosmological parameters | Fisher | the **exact posterior** + SBC |
| 3 | **Diffusion** delensing | the quadratic estimator | the **known input** φ + posterior **coverage** |

If you take away one thing, take this: **never trust a learned estimator you
haven't checked against a known answer** — its point estimate *and* its claimed
uncertainty. We use [CAMB](https://camb.info) for all the physics, so the whole
notebook rests on one well-understood Boltzmann code.

*Assumed background:* you know CMB physics (power spectra, lensing). We explain
the ML as we go.

In [ ]:
# === Setup ===
# On Google Colab this installs the extra packages and downloads the dataset + the
# diffusion checkpoint from the school repo's GitHub Release. On a machine that already
# has them (e.g. a Flatiron node, where torch/numpy/etc. are provided) it is a no-op.
import sys, os
try:
    import camb, sbi, getdist  # noqa: F401  (torch/numpy/scipy are preinstalled on Colab)
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "camb", "sbi", "getdist"],
                   check=True)

if "google.colab" in sys.modules:
    import urllib.request
    _REL = ("https://github.com/flatironinstitute/2026-flatiron-cmb-summer-school"
            "/releases/download/ml-data-v1")
    for _fn in ("camb_dataset.npz", "diffusion_phi.pt"):   # ~720 MB + ~18 MB
        if not os.path.exists(_fn):
            print(f"downloading {_fn} from the GitHub release ...")
            urllib.request.urlretrieve(f"{_REL}/{_fn}", _fn)
    print("Colab setup complete — GPU runtime recommended (Runtime > Change runtime type).")

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

rng = np.random.default_rng(0)
torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch device:", DEVICE)
plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

## Preamble: CAMB and the cosmic-variance yardstick

Our fiducial cosmology is the Planck 2018 best fit (6-parameter $\Lambda$CDM).
We parametrise with the physical densities $\Omega_b h^2,\ \Omega_c h^2$, the
Hubble constant $H_0$, the optical depth $\tau$, the primordial amplitude
$\log(10^{10}A_s)$, and tilt $n_s$.

CAMB returns $D_\ell \equiv \ell(\ell+1)C_\ell/2\pi$ in $\mu K^2$.

In [ ]:
FIDUCIAL = dict(ombh2=0.02237, omch2=0.1200, H0=67.36,
                tau=0.0544, logA=3.044, ns=0.9649)
PARAM_NAMES = ["ombh2", "omch2", "H0", "tau", "logA", "ns"]


def run_camb(p, lmax=2500, lpa=1):
    """Lensed CMB D_ell (muK^2) for a parameter dict p."""
    import camb
    pars = camb.set_params(ombh2=p["ombh2"], omch2=p["omch2"], H0=p["H0"],
                           tau=p["tau"], As=1e-10 * np.exp(p["logA"]), ns=p["ns"])
    pars.set_for_lmax(lmax, lens_potential_accuracy=lpa)
    res = camb.get_results(pars)
    pw = res.get_cmb_power_spectra(pars, CMB_unit="muK", raw_cl=False)
    ls = np.arange(pw["lensed_scalar"].shape[0])
    return dict(ell=ls, TT=pw["lensed_scalar"][:, 0], EE=pw["lensed_scalar"][:, 1],
                BB=pw["lensed_scalar"][:, 2], TE=pw["lensed_scalar"][:, 3])


t0 = time.time()
fid = run_camb(FIDUCIAL)
print(f"One CAMB call took {time.time() - t0:.1f} s  "
      f"(this is exactly why we want an emulator!)")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
ell = fid["ell"]
for a, key, ttl in zip(ax.flat, ["TT", "EE", "TE", "BB"],
                       [r"$D_\ell^{TT}$", r"$D_\ell^{EE}$", r"$D_\ell^{TE}$",
                        r"$D_\ell^{BB}$ (lensing)"]):
    a.plot(ell[2:], fid[key][2:])
    a.set_title(ttl); a.set_xlabel(r"$\ell$"); a.set_ylabel(r"$\mu K^2$")
ax[1, 1].set_yscale("log")
fig.suptitle("Fiducial lensed CMB spectra (CAMB)")
fig.tight_layout()

**The cosmic-variance yardstick.** No measurement of $C_\ell$ can beat the
scatter from having only $(2\ell+1)f_{\rm sky}$ independent modes:
$$\frac{\Delta C_\ell}{C_\ell}=\sqrt{\frac{2}{(2\ell+1)\,f_{\rm sky}}}.$$
We will hold every learned model to this bar: *errors well below cosmic variance
don't matter.* This is the right notion of "good enough" for an emulator.

In [ ]:
def cosmic_variance(ell, fsky=0.4):
    return np.sqrt(2.0 / ((2 * ell + 1) * fsky))

---
# Act 1 — Build a CMB power-spectrum emulator

CAMB takes ~seconds per evaluation. Parameter inference needs $10^4$–$10^6$
evaluations, so we train a neural network to map
$$\boldsymbol\theta=(\Omega_b h^2,\Omega_c h^2,H_0,\tau,\log10^{10}A_s,n_s)
\;\longrightarrow\; D_\ell^{TT,EE,TE},$$
learning from a precomputed table of CAMB spectra. We then **validate against
CAMB** and show the error sits far below cosmic variance — at $\sim\!10^5\times$ the
speed (the exact factor is measured and printed below).

In [ ]:
# Load the precomputed CAMB training set. Prefer the full production dataset
# (generated on a compute node via scripts/generate_camb_dataset.sbatch); fall
# back to the small dev set. On Colab, set DATA_URL to a hosted copy.
DATA_URL = ("https://github.com/flatironinstitute/2026-flatiron-cmb-summer-school"
            "/releases/download/ml-data-v1/camb_dataset.npz")  # used only if no local copy
candidates = ["camb_dataset.npz", "data/camb_dataset.npz", "data/camb_dataset_dev.npz"]
path = next((c for c in candidates if Path(c).exists()), None)
if path is None and DATA_URL:
    import urllib.request
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, "camb_dataset.npz")
    path = "camb_dataset.npz"
assert path, "No dataset found — generate one with scripts/generate_camb_dataset.py"
data = np.load(path, allow_pickle=True)
print("Loaded", path, "with", data["params"].shape[0], "spectra")

ell_e = data["ell"]                       # multipoles 2..2500
params = data["params"].astype(np.float64)  # (N, 6)
low, high = data["param_low"], data["param_high"]
DlTT, DlEE, DlTE = data["Dl_TT"], data["Dl_EE"], data["Dl_TE"]
N = params.shape[0]

### Preprocessing
Two standard tricks make the learning easy:
1. **Transform the targets.** $D_\ell^{TT},D_\ell^{EE}>0$ span orders of
   magnitude, so we learn $\log D_\ell$; $D_\ell^{TE}$ changes sign, so we keep
   it linear.
2. **Compress with PCA.** The ~7500-number spectrum lives on a smooth
   low-dimensional manifold. We keep the top ~64 principal components, so the
   network predicts 64 numbers instead of 7500 — far easier with limited data.

In [ ]:
# Inputs: scale parameters to [0, 1] using the prior box.
X = (params - low) / (high - low)

# Targets: log the positive spectra, keep TE linear, concatenate.
Y = np.concatenate([np.log(DlTT), np.log(DlEE), DlTE], axis=1)  # (N, 3*nell)
nell = ell_e.size

# Train / test split. Shuffle first so the test set is i.i.d., not a biased corner of
# parameter space (in case the table was generated in any systematic order).
ntest = max(8, N // 6)
_perm = np.random.default_rng(0).permutation(N)
itest, itrain = _perm[:ntest], _perm[ntest:]
Xtr, Xte = X[itrain], X[itest]
Ytr, Yte = Y[itrain], Y[itest]

# PCA on the TRAINING targets only (no leakage). The target vector is ~7500-dim,
# so a full SVD is huge; we only need the top ~64 modes, so we use a fast
# randomized SVD (Halko et al. 2011).
Ymean = Ytr.mean(0)
Yc = Ytr - Ymean
n_pca = min(64, Yc.shape[0] - 1)   # rank of mean-centered data is <= n_train - 1


def randomized_svd(M, k, n_oversample=10, seed=0):
    rng = np.random.default_rng(seed)
    Q, _ = np.linalg.qr(M @ rng.standard_normal((M.shape[1], k + n_oversample)))
    _, S, Vt = np.linalg.svd(Q.T @ M, full_matrices=False)
    return S[:k], Vt[:k]


S, basis = randomized_svd(Yc, n_pca)       # basis: (n_pca, 3*nell)
Ctr = Yc @ basis.T                         # (Ntr, n_pca) PCA coefficients
cmean, cstd = Ctr.mean(0), Ctr.std(0)
var_explained = (S ** 2).sum() / (Yc ** 2).sum()
print(f"PCA keeps {n_pca} components; they capture {100 * var_explained:.4f}% of the variance")

# Why does so little PCA capture so much? The spectra depend on only 6 cosmological
# parameters, so they live on a 6-D *manifold* — but it is curved, so a *linear* PCA
# basis needs more than 6 components to span it: the singular values decay smoothly,
# with no sharp cutoff at 6 — the curvature spreads the manifold across many modes.
# 64 is already generous (kept for margin) — which is why a tiny network can emulate
# the whole spectrum.
plt.figure(figsize=(5.5, 3.5))
plt.semilogy(np.arange(1, len(S) + 1), S / S[0], "o-", ms=3)
plt.axvline(6, color="C3", ls="--", lw=1, label="6 parameters (lower bound on rank)")
plt.xlabel("principal component"); plt.ylabel(r"singular value $/\,S_1$")
plt.legend(); plt.title("Spectra live on a (curved) ~6-D manifold")

### The emulator network
A small multilayer perceptron: 6 inputs → a few hidden layers → `n_pca` outputs
(the standardized PCA coefficients).

**Exercise:** fill in the network architecture.

In [ ]:
class Emulator(torch.nn.Module):
    def __init__(self, n_in=6, n_out=n_pca, width=256, depth=3):
        super().__init__()
        # TODO: build an MLP with `depth` hidden layers of `width`, ReLU/GELU activations
        raise NotImplementedError("TODO: build an MLP with `depth` hidden layers of `width`, ReLU/GELU activations")

    def forward(self, x):
        return self.net(x)


# Tensors for the standardized PCA-coefficient targets.
Xtr_t = torch.tensor(Xtr, dtype=torch.float32, device=DEVICE)
Ctr_t = torch.tensor((Ctr - cmean) / cstd, dtype=torch.float32, device=DEVICE)
model = Emulator().to(DEVICE)
print(model)

### Training
**Exercise:** complete the training step (forward → MSE loss → backprop → step).

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)
n_epochs, batch = 400, 128
# Held-out validation set (test PCA coeffs, transformed exactly like train) so we can
# watch train vs. validation loss and catch overfitting -- not just trust the final
# CAMB comparison.
Xte_t = torch.tensor(Xte, dtype=torch.float32, device=DEVICE)
Cte_t = torch.tensor(((Yte - Ymean) @ basis.T - cmean) / cstd, dtype=torch.float32, device=DEVICE)
losses, val_losses = [], []
for epoch in range(n_epochs):
    perm = torch.randperm(Xtr_t.shape[0], device=DEVICE)
    ep_loss = 0.0
    model.train()
    for i in range(0, len(perm), batch):
        idx = perm[i:i + batch]
        # TODO: forward, MSE loss, zero_grad, backward, step
        raise NotImplementedError("TODO: forward, MSE loss, zero_grad, backward, step")
        ep_loss += loss.item() * len(idx)
    losses.append(ep_loss / len(perm))
    model.eval()
    with torch.no_grad():
        val_losses.append(torch.mean((model(Xte_t) - Cte_t) ** 2).item())
    if (epoch + 1) % 50 == 0:
        print(f"epoch {epoch + 1:4d}  train MSE {losses[-1]:.4e}  val MSE {val_losses[-1]:.4e}")

plt.figure()
plt.semilogy(losses, label="train")
plt.semilogy(val_losses, label="validation")
plt.xlabel("epoch"); plt.ylabel("MSE (PCA coeffs)"); plt.legend()
plt.title("Emulator training (watch train vs. validation)")

### Validation against CAMB
The held-out test spectra *are* CAMB. We invert the PCA + log transforms, then
compare emulator vs. CAMB as a **fractional error per multipole**, overlaid on
the cosmic-variance band. The story: the emulator error lives far below the bar
that fundamentally limits any measurement.

In [ ]:
@torch.no_grad()
def emulate(Xn):
    """Parameters in [0,1] -> dict of D_ell predictions."""
    c = model(torch.as_tensor(Xn, dtype=torch.float32, device=DEVICE)).cpu().numpy()
    Yp = (c * cstd + cmean) @ basis + Ymean      # undo standardize + PCA
    tt, ee, te = np.split(Yp, [nell, 2 * nell], axis=-1)
    return dict(TT=np.exp(tt), EE=np.exp(ee), TE=te)


pred = emulate(Xte)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, key in zip(ax, ["TT", "EE"]):
    true = {"TT": DlTT, "EE": DlEE}[key][itest]
    frac = np.abs(pred[key] - true) / np.abs(true)
    med = np.median(frac, axis=0)
    hi = np.percentile(frac, 95, axis=0)
    _i = int(np.argmin(np.abs(ell_e - 1000)))
    print(f"  {key}: median |frac err| / cosmic-variance at l~1000 = "
          f"{med[_i] / cosmic_variance(ell_e)[_i]:.3f}")
    a.fill_between(ell_e, 0, cosmic_variance(ell_e), alpha=0.25,
                   label=r"cosmic variance ($f_{\rm sky}=0.4$, per $\ell$)")
    a.plot(ell_e, med, label="emulator median |frac. err|")
    a.plot(ell_e, hi, lw=0.8, alpha=0.7, label="95th percentile")
    a.set_yscale("log"); a.set_ylim(1e-5, 1)
    a.set_xlabel(r"$\ell$"); a.set_title(f"{key}: emulator vs CAMB")
    a.legend(fontsize=8)
fig.tight_layout()

### Speed-up
The whole point: how much faster is the emulator than CAMB? We time the emulator in
**batched** mode (256 spectra at once), i.e. *amortized throughput per spectrum* —
the number that matters for an MCMC/SBI run that needs $10^5$–$10^6$ evaluations. A
single emulator call carries more overhead (still ~ms), so we print both.

In [ ]:
t0 = time.time()
_ = run_camb(FIDUCIAL)
t_camb = time.time() - t0
xb = np.tile((np.array([FIDUCIAL[k] for k in PARAM_NAMES]) - low) / (high - low), (256, 1))
t0 = time.time()
_ = emulate(xb)
t_emu = (time.time() - t0) / 256
t0 = time.time()
_ = emulate(xb[:1])
t_emu1 = time.time() - t0
print(f"CAMB: {t_camb:.2f} s/spectrum")
print(f"emulator: {t_emu * 1e3:.3f} ms/spectrum batched (256-wide), {t_emu1 * 1e3:.2f} ms single call")
print(f"batched-throughput speed-up: {t_camb / t_emu:.0f}x")

**Try it:** retrain keeping only `n_pca = 8` components — where does the
emulator break first, and is it still below cosmic variance? How many training
spectra do you need to get TT below cosmic variance for $\ell < 2000$?

> With the small *dev* dataset the error bands will be loose; with the full
> production dataset (`generate_camb_dataset.sbatch`) the emulator drops well
> under cosmic variance across the acoustic peaks.

---
# Act 2 — Simulation-based inference of cosmological parameters

Now we use the emulator as a *fast simulator* to infer parameters from a mock
CMB measurement. We infer $(\Omega_c h^2,\ H_0)$ with the other four fixed.
This is the concrete payoff of Act 1: because the emulator is a **differentiable
network**, we get the Fisher Jacobian by autodiff and can sweep an exact posterior
grid in seconds — neither would be practical calling CAMB directly.

**Simulation-based inference (SBI)** learns the posterior $p(\theta\,|\,d)$
directly from (parameter, data) pairs — no explicit likelihood needed. We use
*neural posterior estimation* (NPE): a normalizing flow trained on simulations.

We then **validate three ways**:
1. against a **Fisher** forecast (the Gaussian/linear approximation);
2. against the **exact** posterior (a 2-D grid — tractable here because we adopt
   a Gaussian bandpower likelihood and only have two parameters);
3. with **simulation-based calibration** (SBC), which checks the SBI posterior
   is statistically calibrated.

Because the emulator is *both* the simulator and the "truth," all three methods
share one forward model — so any disagreement is a real statement about the
inference, not about the physics.

In [ ]:
# --- Differentiable, batched emulator pipeline (theta -> binned C_ell) ---------
INFER = ["omch2", "H0"]
idx_infer = [PARAM_NAMES.index(k) for k in INFER]
fid_vec = torch.tensor([FIDUCIAL[k] for k in PARAM_NAMES], dtype=torch.float32, device=DEVICE)

# bandpowers over ell = 30..2000
lmin_b, lmax_b, dlb = 30, 2000, 80
edges = np.arange(lmin_b, lmax_b + dlb, dlb)
nb = len(edges) - 1
ell_b = 0.5 * (edges[1:] + edges[:-1])
# binning matrix Bmat (nb, nell): row-averages C_ell within each band
Bmat = np.zeros((nb, nell))
for b in range(nb):
    m = (ell_e >= edges[b]) & (ell_e < edges[b + 1])
    Bmat[b, m] = 1.0 / m.sum()
Bmat_t = torch.tensor(Bmat, dtype=torch.float32, device=DEVICE)
# Act 1 emulated D_ell = l(l+1)C_l/2pi; here we convert back to C_ell because the Knox
# bandpower covariance below is naturally written in C_ell.
D_to_C_t = torch.tensor((2 * np.pi) / (ell_e * (ell_e + 1.0)), dtype=torch.float32, device=DEVICE)

# torch copies of the PCA/normalization pieces from Act 1
basis_t = torch.tensor(basis, dtype=torch.float32, device=DEVICE)
Ymean_t = torch.tensor(Ymean, dtype=torch.float32, device=DEVICE)
cmean_t = torch.tensor(cmean, dtype=torch.float32, device=DEVICE)
cstd_t = torch.tensor(cstd, dtype=torch.float32, device=DEVICE)
low_t = torch.tensor(low, dtype=torch.float32, device=DEVICE)
high_t = torch.tensor(high, dtype=torch.float32, device=DEVICE)


def theory_Cb(theta2):
    """(B,2) physical (omch2,H0) -> (B,3*nb) binned C_ell [TT,EE,TE], differentiable."""
    B = theta2.shape[0]
    cols = []
    for i in range(6):
        if i in idx_infer:
            cols.append(theta2[:, [idx_infer.index(i)]])
        else:
            cols.append(fid_vec[i].expand(B, 1))
    theta6 = torch.cat(cols, dim=1)
    Xn = (theta6 - low_t) / (high_t - low_t)
    Y = (model(Xn) * cstd_t + cmean_t) @ basis_t + Ymean_t
    logTT, logEE, TE = Y[:, :nell], Y[:, nell:2 * nell], Y[:, 2 * nell:]
    ClTT = torch.exp(logTT) * D_to_C_t
    ClEE = torch.exp(logEE) * D_to_C_t
    ClTE = TE * D_to_C_t
    return torch.cat([ClTT @ Bmat_t.T, ClEE @ Bmat_t.T, ClTE @ Bmat_t.T], dim=1)

In [ ]:
# --- Knox bandpower covariance from the fiducial spectra ----------------------
FSKY = 0.4
NOISE_T = 15.0  # uK-arcmin (white); N_EE = 2 N_TT
nlT = (NOISE_T * np.pi / 180 / 60) ** 2
nlP = 2 * nlT

with torch.no_grad():
    cb_fid = theory_Cb(fid_vec[idx_infer].unsqueeze(0)).cpu().numpy()[0]
Tb, Eb, Xb = cb_fid[:nb], cb_fid[nb:2 * nb], cb_fid[2 * nb:]
Ttot, Etot = Tb + nlT, Eb + nlP
nmodes = np.array([FSKY * np.sum(2 * ell_e[(ell_e >= edges[b]) & (ell_e < edges[b + 1])] + 1)
                   for b in range(nb)])

Sigma = np.zeros((3 * nb, 3 * nb))
for b in range(nb):
    iT, iE, iX = b, nb + b, 2 * nb + b
    n = nmodes[b]
    Sigma[iT, iT] = 2 * Ttot[b] ** 2 / n
    Sigma[iE, iE] = 2 * Etot[b] ** 2 / n
    Sigma[iX, iX] = (Xb[b] ** 2 + Ttot[b] * Etot[b]) / n
    Sigma[iT, iE] = Sigma[iE, iT] = 2 * Xb[b] ** 2 / n
    Sigma[iT, iX] = Sigma[iX, iT] = 2 * Ttot[b] * Xb[b] / n
    Sigma[iE, iX] = Sigma[iX, iE] = 2 * Etot[b] * Xb[b] / n
Sigma_inv = np.linalg.inv(Sigma)
L_sigma = np.linalg.cholesky(Sigma)
print(f"{nb} bandpowers x 3 spectra = {3 * nb}-dim data vector; fsky={FSKY}")

In [ ]:
# --- A mock observation at the fiducial parameters ----------------------------
theta_true = np.array([FIDUCIAL["omch2"], FIDUCIAL["H0"]])
with torch.no_grad():
    mean_true = theory_Cb(torch.tensor(theta_true[None], dtype=torch.float32, device=DEVICE)).cpu().numpy()[0]
# Mock data = the noiseless fiducial expectation (the "Asimov" dataset). This puts
# SBI, the exact grid, and Fisher at the same central point, so we compare posterior
# *shapes* without the scatter of one particular noise draw. To explore off-centre
# behaviour, add a noise term:
#     d_obs = mean_true + L_sigma @ np.random.default_rng(1).standard_normal(3 * nb)
d_obs = mean_true.copy()

plt.figure()
plt.errorbar(ell_b, ell_b ** 2 * d_obs[:nb], yerr=ell_b ** 2 * np.sqrt(np.diag(Sigma)[:nb]),
             fmt="o", ms=3, label=r"fiducial $C_\ell^{TT}\pm$ Knox errors")
plt.xlabel(r"$\ell$"); plt.ylabel(r"$\ell^2 C_\ell^{TT}$"); plt.legend(); plt.title("Mock TT bandpowers")

### The simulator and NPE
The simulator draws a parameter, computes the binned spectra with the emulator,
and adds a Knox-covariance noise realization. We train an amortized NPE
(normalizing flow) on a few thousand such simulations.

**The prior is yours to choose.** Below it's uniform and broad (its width in
Fisher-$\sigma$ is printed in Validation 1&2) — broad enough to be uninformative,
narrow enough that the simulations stay dense where the data live. **Try it:** (1) add a noise term to `d_obs` (above) so the data sit
*off-centre* — the exact posterior and a well-trained SBI both shift to follow it,
while Fisher stays at the input. (2) Then widen the prior toward the full emulator
box (`[0.10, 0.14] × [60, 76]`): with the same number of simulations now spread
over a far larger volume, the amortized SBI gets noisier and drifts from the exact
posterior, most visibly off-centre. Too *narrow* a prior instead truncates it.
Tuning the prior (and the simulation count) is part of the craft of SBI.

**Exercise:** complete the simulator (emulator mean + correlated noise).

In [ ]:
from sbi.inference import NPE
from sbi.utils import BoxUniform

# Prior: uniform and uninformative, but not absurdly wide -- a broad box around the
# truth (still well inside the emulator's training range; its width in Fisher-sigma is
# printed in Validation 1&2). A box far wider than the posterior spreads the
# simulations thin and degrades the amortized NPE, especially away from the centre.
pr_lo = torch.tensor([0.114, 63.0], dtype=torch.float32)
pr_hi = torch.tensor([0.126, 72.0], dtype=torch.float32)
prior = BoxUniform(low=pr_lo, high=pr_hi)
L_sigma_t = torch.tensor(L_sigma, dtype=torch.float32, device=DEVICE)


def simulator(theta2):
    theta2 = theta2.to(DEVICE)
    # TODO: emulator mean + correlated Gaussian noise (L_sigma @ z)
    raise NotImplementedError("TODO: emulator mean + correlated Gaussian noise (L_sigma @ z)")

In [ ]:
n_sims = 15000
theta_sims = prior.sample((n_sims,))
x_sims = simulator(theta_sims)
# Train the flow on CPU: it's tiny/fast here, and avoids sbi's GPU device juggling.
# (The simulator still calls the GPU emulator internally, so simulating is fast.)
npe = NPE(prior=prior, density_estimator="maf", device="cpu")
npe.append_simulations(theta_sims, x_sims).train()
posterior = npe.build_posterior()
sbi_samples = posterior.sample((20000,), x=torch.tensor(d_obs, dtype=torch.float32)).numpy()
print("trained NPE; drew", sbi_samples.shape[0], "posterior samples")

### Validation 1 & 2: exact grid posterior and Fisher
The exact posterior on the 2-D grid (Gaussian likelihood) and the Fisher ellipse
(linearized model). We overlay all three. **Note:** Fisher uses the *likelihood
only* (no prior), while the exact grid and SBI are truncated to the prior box — they
agree here only because the box is loose (see the printed Fisher-$\sigma$ widths). In
the Try-it above, narrow the prior toward the posterior and Fisher (which ignores the
box) will visibly disagree with the now prior-dominated exact/SBI posteriors.

In [ ]:
# Fisher matrix via autodiff of the mean at the truth (compute first, to size the grid)
th = torch.tensor(theta_true, dtype=torch.float32, device=DEVICE, requires_grad=False)
J = torch.autograd.functional.jacobian(
    lambda t: theory_Cb(t.unsqueeze(0))[0], th).cpu().numpy()  # (3nb, 2)
Fisher = J.T @ Sigma_inv @ J
Cov_fisher = np.linalg.inv(Fisher)

# How wide is the (uniform) prior box in Fisher-sigma units? Fisher uses the LIKELIHOOD
# ONLY (no prior), while the exact grid and SBI are truncated to the box -- so they
# agree only because the box is loose. We print the actual widths instead of asserting
# them (the H0 box is slightly asymmetric, so we show the - and + margins separately).
_sig_f = np.sqrt(np.diag(Cov_fisher))
_tt = np.array(theta_true)
print("prior box half-widths in Fisher-sigma:  "
      f"omch2 [-{(_tt[0]-pr_lo[0].item())/_sig_f[0]:.0f}, +{(pr_hi[0].item()-_tt[0])/_sig_f[0]:.0f}],  "
      f"H0 [-{(_tt[1]-pr_lo[1].item())/_sig_f[1]:.0f}, +{(pr_hi[1].item()-_tt[1])/_sig_f[1]:.0f}]  "
      "(loose box -> Fisher ~ exact ~ SBI)")

# Exact posterior on a grid centered on the truth and sized to the Fisher width
# (so it finely resolves the posterior), clipped to the prior box.
ng = 200
sig = np.sqrt(np.diag(Cov_fisher))
g1 = np.linspace(max(float(pr_lo[0]), theta_true[0] - 6 * sig[0]),
                 min(float(pr_hi[0]), theta_true[0] + 6 * sig[0]), ng)
g2 = np.linspace(max(float(pr_lo[1]), theta_true[1] - 6 * sig[1]),
                 min(float(pr_hi[1]), theta_true[1] + 6 * sig[1]), ng)
G1, G2 = np.meshgrid(g1, g2)
grid = np.stack([G1.ravel(), G2.ravel()], 1)
with torch.no_grad():
    mu = theory_Cb(torch.tensor(grid, dtype=torch.float32, device=DEVICE)).cpu().numpy()
resid = mu - d_obs
chi2 = np.einsum("ij,jk,ik->i", resid, Sigma_inv, resid)
post = np.exp(-0.5 * (chi2 - chi2.min())).reshape(ng, ng)

In [ ]:
# Compare all three with getdist (it auto-zooms to the posterior): the exact grid
# posterior, the SBI samples, and the Fisher Gaussian (drawn as samples).
from getdist import MCSamples, plots

_names, _labels = ["omch2", "H0"], [r"\Omega_c h^2", "H_0"]
_rng = np.random.default_rng(0)
fisher_draws = _rng.multivariate_normal(theta_true, Cov_fisher, size=40000)
_p = (post / post.sum()).ravel()
_idx = _rng.choice(_p.size, size=40000, p=_p)               # resample the grid posterior
_dg = np.array([g1[1] - g1[0], g2[1] - g2[0]])              # + grid-cell jitter so the KDE is well-behaved
exact_draws = np.column_stack([G1.ravel()[_idx], G2.ravel()[_idx]]) + _rng.normal(0, _dg, size=(40000, 2))
mc = [MCSamples(samples=exact_draws, names=_names, labels=_labels, label="exact posterior"),
      MCSamples(samples=sbi_samples, names=_names, labels=_labels, label="SBI (NPE)"),
      MCSamples(samples=fisher_draws, names=_names, labels=_labels, label="Fisher")]
g = plots.get_subplot_plotter(width_inch=7)
g.triangle_plot(mc, filled=[True, False, False], contour_colors=["k", "C0", "C3"],
                markers={"omch2": theta_true[0], "H0": theta_true[1]})

### Validation 3: simulation-based calibration (SBC)
If the SBI posterior is calibrated, then for data drawn from the prior the rank
of the true parameter among posterior samples is *uniform*. We check the rank
histograms are flat. **Caveat:** with only `n_sbc=200` trials over 10 bins, each
bin has ~Poisson scatter of $\pm\sqrt{20}\approx4.5$ (shaded band) — so don't
over-read bin-to-bin wiggles; a *definitive* SBC needs many more trials.

In [ ]:
n_sbc, n_post = 200, 99   # ranks take n_post+1 = 100 values -> exactly 10 per bin (no edge bias)
theta_sbc = prior.sample((n_sbc,))
x_sbc = simulator(theta_sbc)
ranks = np.zeros((n_sbc, 2), dtype=int)
for i in range(n_sbc):
    s = posterior.sample((n_post,), x=x_sbc[i], show_progress_bars=False).numpy()
    ranks[i] = (s < theta_sbc[i].numpy()).sum(0)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
exp = n_sbc / 10
for k, a in enumerate(ax):
    a.hist(ranks[:, k], bins=10, color="steelblue", edgecolor="w")
    a.axhline(exp, color="k", ls="--", label="uniform")
    a.axhspan(exp - exp ** 0.5, exp + exp ** 0.5, color="k", alpha=0.12,
              label=fr"Poisson $\pm\sqrt{{{int(exp)}}}$")
    a.set_title(f"SBC ranks: {INFER[k]}"); a.set_xlabel("rank"); a.legend(fontsize=8)
fig.tight_layout()

**Takeaways.** SBI reproduces the exact posterior, and — because $(\Omega_c h^2,
H_0)$ are well constrained and the model is near-linear here — the Fisher ellipse
too. All three agreeing *is* the validation: the learned posterior is correct, and
SBI never *evaluated* a likelihood (though a Gaussian one exists here — which is
exactly what lets us grade it); the flat SBC ranks confirm it's calibrated.
SBI's real edge shows up when the posterior is non-Gaussian (e.g. parameters near
a physical boundary), where a single Gaussian ellipse can't capture the true shape.

---
# Act 3 — Delensing with a diffusion model (vs. the quadratic estimator)

We now switch from *spectra* to *maps*. Gravitational lensing remaps the CMB by
the deflection $\mathbf d=\nabla\phi$ of the lensing potential $\phi$, smoothing
the acoustic peaks and converting $E$-modes into $B$-modes. **Delensing** means
undoing this — and the first step is reconstructing $\phi$ from the observed
maps.

- The classical tool is the **quadratic estimator (QE)**: a clever quadratic
  combination of the maps (Hu & Okamoto 2002).
- The learned tool is a **conditional diffusion model** that *samples*
  $p(\phi\,|\,\text{maps})$ — giving not just a point estimate but a posterior.

We compare them with the **cross-correlation coefficient** with the true $\phi$,
$\rho_L = C_L^{\hat\phi\phi}/\sqrt{C_L^{\hat\phi\hat\phi}C_L^{\phi\phi}}$, then
use $\hat\phi$ to delens the $B$-modes. (We write capital $L$ for the *lensing*
multipole of $\phi$, to distinguish it from the CMB multipole $\ell$.)

Everything is **flat-sky** and pure numpy/torch so the mechanics are visible.
(Production pipelines use full-sky tools like `pixell`/`lenspyx`; this captures
the ideas.)

### A minimal flat-sky toolkit
Gaussian random fields from a $C_\ell$, $E/B\leftrightarrow Q/U$ rotation, and
lensing by remapping with $\nabla\phi$. The one subtlety worth seeing: lensing
is literally "look up the unlensed sky at the deflected position."

In [ ]:
from numpy.fft import fft2, ifft2
from scipy.ndimage import map_coordinates


class FlatSky:
    def __init__(self, N, L_deg):
        self.N, self.L = N, np.deg2rad(L_deg)
        self.dx = self.L / N
        self.area = self.L ** 2
        l1d = 2 * np.pi * np.fft.fftfreq(N, d=self.dx)
        self.lx, self.ly = np.meshgrid(l1d, l1d)
        self.ell = np.sqrt(self.lx ** 2 + self.ly ** 2)
        self.psi = np.arctan2(self.ly, self.lx)

    def cl_to_2d(self, ell_th, cl_th):
        c = np.interp(self.ell, ell_th, cl_th, left=0.0, right=0.0)
        c[0, 0] = 0.0
        return c

    def grf(self, cl2d, rng):
        w = rng.standard_normal((self.N, self.N))
        return ifft2(fft2(w) * np.sqrt(cl2d)).real / self.dx

    def grf_TE(self, clTT, clTE, clEE, rng):
        wT = fft2(rng.standard_normal((self.N, self.N)))
        wE = fft2(rng.standard_normal((self.N, self.N)))
        sT = np.sqrt(clTT)
        with np.errstate(divide="ignore", invalid="ignore"):
            a = np.where(clTT > 0, clTE / np.where(clTT > 0, sT, 1), 0.0)
            r = np.clip(clEE - np.where(clTT > 0, clTE ** 2 / np.where(clTT > 0, clTT, 1), 0.0), 0, None)
        return (ifft2(wT * sT).real / self.dx,
                ifft2(wT * a + wE * np.sqrt(r)).real / self.dx)

    def measure_cl(self, fa, fb=None, nbins=20, lmin=40, lmax=2000):
        Fa = fft2(fa) * self.dx ** 2
        Fb = Fa if fb is None else fft2(fb) * self.dx ** 2
        p = (Fa * np.conj(Fb)).real / self.area
        edges = np.linspace(lmin, lmax, nbins + 1)
        which = np.digitize(self.ell.ravel(), edges) - 1
        cl, lc = np.zeros(nbins), np.zeros(nbins)
        pr, lr = p.ravel(), self.ell.ravel()
        for b in range(nbins):
            m = which == b
            if m.any():
                cl[b], lc[b] = pr[m].mean(), lr[m].mean()
        ok = lc > 0
        return lc[ok], cl[ok]

    def eb_to_qu(self, E, B):
        Ef, Bf = fft2(E), fft2(B)
        c, s = np.cos(2 * self.psi), np.sin(2 * self.psi)
        return ifft2(c * Ef - s * Bf).real, ifft2(s * Ef + c * Bf).real

    def qu_to_eb(self, Q, U):
        Qf, Uf = fft2(Q), fft2(U)
        c, s = np.cos(2 * self.psi), np.sin(2 * self.psi)
        return ifft2(c * Qf + s * Uf).real, ifft2(-s * Qf + c * Uf).real

    def lens(self, field, phi, order=3):
        phif = fft2(phi)
        dx = ifft2(1j * self.lx * phif).real
        dy = ifft2(1j * self.ly * phif).real
        ny, nx = field.shape
        yy, xx = np.meshgrid(np.arange(ny), np.arange(nx), indexing="ij")
        coords = np.array([yy + dy / self.dx, xx + dx / self.dx])
        return map_coordinates(field, coords, order=order, mode="wrap")

    def qe_tt(self, Tmap, clTT_2d, cltot_2d):
        """Unnormalised TT estimator of phi (normalisation calibrated on sims)."""
        Tf = fft2(Tmap)
        with np.errstate(divide="ignore", invalid="ignore"):
            wiener = np.where(cltot_2d > 0, clTT_2d / cltot_2d, 0.0)
            invvar = np.where(cltot_2d > 0, 1.0 / cltot_2d, 0.0)
        h = ifft2(Tf * invvar).real
        gx = ifft2(1j * self.lx * wiener * Tf).real
        gy = ifft2(1j * self.ly * wiener * Tf).real
        return ifft2(-1j * (self.lx * fft2(gx * h) + self.ly * fft2(gy * h))).real

    def qe_eb(self, Emap, Bmap, clEE_2d, cltotEE_2d, cltotBB_2d):
        """Unnormalised EB estimator of phi -- the polarization lensing estimator,
        much more sensitive than TT at low noise. Weight (Hu & Okamoto 2002, with
        unlensed C^BB~0): f^EB = C^EE_{l1}(L.l1) sin(2(psi_{l1}-psi_{l2})), where the
        two CMB wavevectors sum to the lensing mode L = l1 + l2 (so l2 = L - l1)."""
        Ef, Bf = fft2(Emap), fft2(Bmap)
        c2, s2 = np.cos(2 * self.psi), np.sin(2 * self.psi)
        with np.errstate(divide="ignore", invalid="ignore"):
            wE = np.where(cltotEE_2d > 0, clEE_2d / cltotEE_2d, 0.0)
            iB = np.where(cltotBB_2d > 0, 1.0 / cltotBB_2d, 0.0)
        qc, qs = ifft2(c2 * Bf * iB), ifft2(s2 * Bf * iB)
        num = np.zeros_like(Ef)
        for li in (self.lx, self.ly):
            ps = ifft2(s2 * li * wE * Ef)
            pc = ifft2(c2 * li * wE * Ef)
            num = num + li * (fft2(ps * qc) - fft2(pc * qs))
        return ifft2(num).real

### Fiducial spectra and the patch
One CAMB call gives the unlensed CMB, the lensing potential, and a small
primordial $B$-mode ($r=0.01$). The patch is $256^2$ over $10^\circ$
(pixel $\approx 2.3'$) — fine enough that the $\sim 2.5'$ lensing deflections are
resolved, which the QE needs.

In [ ]:
def fiducial_spectra(lmax=3000, r=0.01):
    import camb
    pars = camb.set_params(ombh2=0.02237, omch2=0.12, H0=67.36, tau=0.0544,
                           As=1e-10 * np.exp(3.044), ns=0.9649, r=r)
    pars.WantTensors = True
    pars.set_for_lmax(lmax, lens_potential_accuracy=1)
    res = camb.get_results(pars)
    pw = res.get_cmb_power_spectra(pars, CMB_unit="muK", raw_cl=True)
    unl, lensed, tens = pw["unlensed_scalar"], pw["lensed_scalar"], pw["tensor"]
    ls = np.arange(unl.shape[0])
    pp = res.get_lens_potential_cls(lmax)
    lp = np.arange(pp.shape[0])
    with np.errstate(divide="ignore", invalid="ignore"):
        clpp = np.where(lp >= 1, pp[:, 0] * 2 * np.pi / (lp * (lp + 1.0)) ** 2, 0.0)
    return dict(ell=ls, TT=unl[:, 0], EE=unl[:, 1], TE=unl[:, 3],
                TT_len=lensed[:, 0], EE_len=lensed[:, 1], BB_len=lensed[:, 2],
                BB_tensor=tens[:, 2], lp=lp, clpp=clpp)


sp = fiducial_spectra()
fs = FlatSky(N=256, L_deg=10.0)
NOISE_T = 2.0  # uK-arcmin (S4-like, much deeper than Act 2's 15; pol noise = 2x)
nlT = (NOISE_T * np.pi / 180 / 60) ** 2

TT2d = fs.cl_to_2d(sp["ell"], sp["TT"])        # unlensed: used to generate skies
TTlen2d = fs.cl_to_2d(sp["ell"], sp["TT_len"])  # lensed: used in the QE filter
EE2d = fs.cl_to_2d(sp["ell"], sp["EE"])
TE2d = fs.cl_to_2d(sp["ell"], sp["TE"])
BB2d = fs.cl_to_2d(sp["ell"], sp["BB_tensor"])
pp2d = fs.cl_to_2d(sp["lp"], sp["clpp"])
# band-limit the QE to CMB modes we trust (crucial: avoids amplifying empty modes)
band = (fs.ell >= 30) & (fs.ell <= 3000)
TT_grad = np.where(band, TTlen2d, 0.0)
cltot2d = np.where(band, TTlen2d + nlT, np.inf)
# EB QE filters: gradient weight = unlensed EE; totals = lensed + pol noise (2*nlT)
EElen2d = fs.cl_to_2d(sp["ell"], sp["EE_len"])
BBlen2d = fs.cl_to_2d(sp["ell"], sp["BB_len"])
EE_grad = np.where(band, EE2d, 0.0)
cltotEE = np.where(band, EElen2d + 2 * nlT, np.inf)
cltotBB = np.where(band, BBlen2d + 2 * nlT, np.inf)

In [ ]:
def simulate(rng, with_pol=False):
    """One lensed sky. Returns dict with phi, lensed T (+noise), optionally E,B."""
    phi = fs.grf(pp2d, rng)
    out = {"phi": phi}
    if not with_pol:
        Tu = fs.grf(TT2d, rng)
        out["T"] = fs.lens(Tu, phi) + fs.grf(np.full_like(TT2d, nlT), rng)
        return out
    Tu, Eu = fs.grf_TE(TT2d, TE2d, EE2d, rng)
    Bu = fs.grf(BB2d, rng)                      # primordial B (r)
    Q, U = fs.eb_to_qu(Eu, Bu)
    Tl = fs.lens(Tu, phi)
    Ql, Ul = fs.lens(Q, phi), fs.lens(U, phi)
    El, Bl = fs.qu_to_eb(Ql, Ul)
    out.update(T=Tl + fs.grf(np.full_like(TT2d, nlT), rng),
               E=El + fs.grf(np.full_like(TT2d, 2 * nlT), rng),
               B=Bl + fs.grf(np.full_like(TT2d, 2 * nlT), rng),
               B_lensing=Bl, B_prim=Bu)
    return out


rng = np.random.default_rng(0)
ex = simulate(rng, with_pol=True)
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
for a, key, t in zip(ax, ["phi", "T", "B"],
                     [r"true $\phi$", "lensed T + noise", "observed B"]):
    im = a.imshow(ex[key], cmap="RdBu_r"); a.set_title(t); a.axis("off")
    fig.colorbar(im, ax=a, fraction=0.046)
fig.tight_layout()

### The quadratic estimators (TT and EB)
The QE correlates two filtered copies of the maps. We build **two** baselines:
the **TT** estimator (temperature only — a weak baseline) and the **EB** estimator
(polarization, the lensing-optimal estimator at low noise — and **the same data our
diffusion sees**, so it is the *fair* foil for a B-mode story). We know the input
$\phi$ in our sims, so we calibrate on simulations and quote the cross-correlation
$\rho_L$ with the truth (normalization-independent).

In [ ]:
nsim_qe = 30
bin_kw = dict(lmin=40, lmax=600, nbins=10)
qe_tt_list, qe_eb_list, phi_list = [], [], []
for _ in range(nsim_qe):
    s = simulate(rng, with_pol=True)
    qe_tt_list.append(fs.qe_tt(s["T"], TT_grad, cltot2d))
    qe_eb_list.append(fs.qe_eb(s["E"], s["B"], EE_grad, cltotEE, cltotBB))
    phi_list.append(s["phi"])


def _rho(est_list):
    return np.mean([fs.measure_cl(a, b, **bin_kw)[1] /
                    np.sqrt(fs.measure_cl(a, **bin_kw)[1] * fs.measure_cl(b, **bin_kw)[1])
                    for a, b in zip(est_list, phi_list)], axis=0)


rho_qe = _rho(qe_tt_list)   # temperature QE (weak baseline)
rho_eb = _rho(qe_eb_list)   # polarization EB QE (same E,B data as the diffusion -> fair foil)
Lb = fs.measure_cl(qe_tt_list[0], phi_list[0], **bin_kw)[0]
print("TT QE rho_L:", np.array2string(rho_qe, precision=2))
print("EB QE rho_L:", np.array2string(rho_eb, precision=2))

### The diffusion model
**Diffusion in one paragraph:** take a clean field $\phi$ and add Gaussian noise in
many small steps until it is pure noise (the *forward* process); train a network to
predict the noise added at a random step; then *generate* by starting from noise and
undoing the steps. Let the network also see a **conditioning** input and it learns
the *conditional* distribution $p(\phi\,|\,\text{cond})$ — exactly what delensing needs.

Here a conditional DDPM learns $\phi$ given the **polarization maps** $(E,B)$ —
*the same data the EB QE uses*, so the comparison below is apples-to-apples. We
generate $(E, B, \phi)$ training pairs, train a small U-Net, then **sample** $\phi$
for held-out maps.

> **GPU strongly recommended** (Colab: choose a GPU runtime). Training ~10–20 min.
> With `LOAD_CHECKPOINT=True` (default) the training pairs aren't generated at all —
> the checkpoint carries the weights *and* the normalizations, so only the (fast)
> sampling below runs.

In [ ]:
import os
import torch.nn as nn
import torch.nn.functional as Fnn


def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t.device) / half)
    args = t[:, None].float() * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)


class ResBlock(nn.Module):
    def __init__(self, cin, cout, temb_dim):
        super().__init__()
        g = lambda c: min(8, c)
        self.norm1, self.conv1 = nn.GroupNorm(g(cin), cin), nn.Conv2d(cin, cout, 3, padding=1)
        self.temb = nn.Linear(temb_dim, cout)
        self.norm2, self.conv2 = nn.GroupNorm(g(cout), cout), nn.Conv2d(cout, cout, 3, padding=1)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, temb):
        h = self.conv1(Fnn.silu(self.norm1(x)))
        h = h + self.temb(temb)[:, :, None, None]
        h = self.conv2(Fnn.silu(self.norm2(h)))
        return h + self.skip(x)


class UNet(nn.Module):
    def __init__(self, cond_ch=1, base=48, mults=(1, 2, 2, 2), temb_dim=128):
        super().__init__()
        self.temb_dim = temb_dim
        self.temb = nn.Sequential(nn.Linear(temb_dim, temb_dim), nn.SiLU(),
                                  nn.Linear(temb_dim, temb_dim))
        chs = [base * m for m in mults]
        self.in_conv = nn.Conv2d(1 + cond_ch, base, 3, padding=1)
        self.downs, cin = nn.ModuleList(), base
        for c in chs:
            self.downs.append(ResBlock(cin, c, temb_dim))
            self.downs.append(nn.Conv2d(c, c, 4, stride=2, padding=1)); cin = c
        self.mid = ResBlock(cin, cin, temb_dim)
        self.ups = nn.ModuleList()
        for c in reversed(chs):
            self.ups.append(nn.ConvTranspose2d(cin, c, 4, stride=2, padding=1))
            self.ups.append(ResBlock(2 * c, c, temb_dim)); cin = c
        self.out_norm = nn.GroupNorm(min(8, cin), cin)
        self.out_conv = nn.Conv2d(cin, 1, 3, padding=1)

    def forward(self, x, cond, t):
        temb = self.temb(timestep_embedding(t, self.temb_dim))
        h = self.in_conv(torch.cat([x, cond], 1))
        skips = [h]
        for i in range(0, len(self.downs), 2):
            h = self.downs[i](h, temb); skips.append(h); h = self.downs[i + 1](h)
        h = self.mid(h, temb)
        for i in range(0, len(self.ups), 2):
            h = self.ups[i](h); h = self.ups[i + 1](torch.cat([h, skips.pop()], 1), temb)
        return self.out_conv(Fnn.silu(self.out_norm(h)))


class DDPM:
    def __init__(self, model, n_steps=1000, device=DEVICE):
        self.model, self.device, self.T = model.to(device), device, n_steps
        self.betas = torch.linspace(1e-4, 0.02, n_steps, device=device)
        self.abar = torch.cumprod(1 - self.betas, 0)

    def loss(self, phi, cond):
        # TODO: the forward (noising) process + the training target
        raise NotImplementedError("TODO: the forward (noising) process + the training target")

    @torch.no_grad()
    def sample(self, cond, n_steps=100, eta=1.0):
        # DDIM sampler with stochasticity `eta`: eta=0 is deterministic, eta=1 is
        # (ancestral) DDPM-like. We use eta=1 so the *spread* of samples is a real
        # posterior, not just a deterministic map of the initial noise -- this is what
        # we validate with the coverage test below.
        ts = torch.linspace(self.T - 1, 0, n_steps, device=self.device).long()
        x = torch.randn(cond.shape[0], 1, cond.shape[2], cond.shape[3], device=self.device)
        for i, t in enumerate(ts):
            tb = torch.full((cond.shape[0],), int(t), device=self.device)
            ab = self.abar[t]
            pred = self.model(x, cond, tb)
            x0 = (x - (1 - ab).sqrt() * pred) / ab.sqrt()
            if i < len(ts) - 1:
                abn = self.abar[ts[i + 1]]
                sigma = eta * ((1 - abn) / (1 - ab)).sqrt() * (1 - ab / abn).sqrt()
                x = (abn.sqrt() * x0 + (1 - abn - sigma ** 2).clamp(min=0).sqrt() * pred
                     + sigma * torch.randn_like(x))
            else:
                x = x0
        return x

In [ ]:
LOAD_CHECKPOINT = True   # use the provided checkpoint; set False to train live (~10 min on a GPU)
CKPT = next((p for p in ["diffusion_phi.pt", "data/diffusion_phi.pt"]
             if os.path.exists(p)), "diffusion_phi.pt")
ck = torch.load(CKPT, map_location=DEVICE) if (LOAD_CHECKPOINT and os.path.exists(CKPT)) else None
base = ck["base"] if (ck and "base" in ck) else 64          # match the checkpoint's width
ddpm = DDPM(UNet(cond_ch=2, base=base, mults=(1, 2, 2, 2)),
            n_steps=(ck["n_steps"] if (ck and "n_steps" in ck) else 1000), device=DEVICE)
if ck is not None:
    ddpm.model.load_state_dict(ck["state_dict"] if "state_dict" in ck else ck)
    E_std, B_std, P_std = ck["E_std"], ck["B_std"], ck["P_std"]  # checkpoint's exact norms
    print("loaded checkpoint", CKPT)
else:
    # Train live. We synthesize the (E,B,phi) pairs HERE, inside the train branch, so
    # the (expensive) sim generation is skipped entirely on the default checkpoint path.
    # NOTE: this uses 3000 sims / 40 epochs for speed -- the provided checkpoint used
    # 15000 / 150, so a live-trained model will UNDERPERFORM the plotted curves.
    if DEVICE == "cpu":
        print("WARNING: no GPU — training will be very slow. Use a GPU runtime or a checkpoint.")
    n_train = 3000
    Etr = np.empty((n_train, fs.N, fs.N), np.float32)
    Btr = np.empty((n_train, fs.N, fs.N), np.float32)
    Ptr = np.empty((n_train, fs.N, fs.N), np.float32)
    for i in range(n_train):
        s = simulate(rng, with_pol=True)            # condition on (E,B) -- same data as the EB QE
        Etr[i], Btr[i], Ptr[i] = s["E"], s["B"], s["phi"]
    E_std, B_std, P_std = Etr.std(), Btr.std(), Ptr.std()
    cond_tr = torch.tensor(np.stack([Etr / E_std, Btr / B_std], axis=1), device=DEVICE)  # (n,2,H,W)
    phi_tr = torch.tensor(Ptr[:, None] / P_std, device=DEVICE)
    opt = torch.optim.Adam(ddpm.model.parameters(), lr=2e-4)
    for ep in range(40):
        perm = torch.randperm(n_train, device=DEVICE)
        tot = 0.0
        for j in range(0, n_train, 32):
            idx = perm[j:j + 32]
            loss = ddpm.loss(phi_tr[idx], cond_tr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        if (ep + 1) % 5 == 0:
            print(f"epoch {ep + 1:3d}  loss {tot / n_train:.4f}")
    torch.save({"state_dict": ddpm.model.state_dict(), "cond": "EB", "cond_ch": 2,
                "E_std": float(E_std), "B_std": float(B_std), "P_std": float(P_std),
                "base": base, "n_steps": ddpm.T}, CKPT)

### Sample the posterior and validate against the QE
The diffusion model returns *samples*: many plausible $\phi$ for one map. Their
mean is the reconstruction; their scatter is the uncertainty (the QE only gives
this through its analytic $N_0$).

In [ ]:
n_test = 20
test = [simulate(rng, with_pol=True) for _ in range(n_test)]
cond_te = torch.tensor(
    np.stack([np.stack([t["E"] / E_std, t["B"] / B_std]) for t in test]),
    dtype=torch.float32, device=DEVICE)            # (n_test, 2, H, W)
# Draw a full set of posterior samples per test map (eta=1 -> a genuine posterior,
# not a deterministic map of the seed). The MEAN is the MMSE point estimate (its
# correlation with truth is the headline below); the SPREAD we validate afterwards.
n_post = 64   # posterior draws per map; also sets the coverage test's finite-sample floor
with torch.no_grad():
    samples = torch.stack([ddpm.sample(cond_te, n_steps=100, eta=1.0).cpu()
                           for _ in range(n_post)])      # (n_post, n_test, 1, H, W)
phi_samples = samples[:, :, 0].numpy() * P_std            # (n_post, n_test, H, W)
phi_diff = phi_samples.mean(0)                            # (n_test, H, W) posterior mean

rho_diff = np.mean([fs.measure_cl(phi_diff[i], test[i]["phi"], **bin_kw)[1] /
                    np.sqrt(fs.measure_cl(phi_diff[i], **bin_kw)[1] *
                            fs.measure_cl(test[i]["phi"], **bin_kw)[1])
                    for i in range(n_test)], axis=0)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].imshow(test[0]["phi"], cmap="RdBu_r"); ax[0].set_title(r"true $\phi$"); ax[0].axis("off")
ax[1].imshow(phi_diff[0], cmap="RdBu_r")
ax[1].set_title(r"diffusion $\hat\phi$ (posterior mean)"); ax[1].axis("off")
fig.tight_layout()

plt.figure()
plt.plot(Lb, rho_qe, "o-", color="C0", label="QE: temperature (TT)")
plt.plot(Lb, rho_eb, "^-", color="C2", label="QE: polarization (EB)")
plt.plot(Lb, rho_diff, "s-", color="C3", label="diffusion (E,B, posterior mean)")
plt.xlabel("L"); plt.ylabel(r"$\rho_L$ with true $\phi$")
plt.ylim(0, 1); plt.legend(); plt.title("Lensing reconstruction: diffusion vs QE (TT and EB)")
print("TT QE     rho_L:", np.round(rho_qe, 2))
print("EB QE     rho_L:", np.round(rho_eb, 2))
print("diffusion rho_L:", np.round(rho_diff, 2))

### Read this gap honestly
This is a **fair fight**: the diffusion and the **EB** QE (green) see *exactly the
same data* $(E,B)$, so the comparison isolates the *method*, not the information.
(The TT QE, blue, is only the weak temperature-only baseline.) And the result is
**not** a blowout — it has a **crossover**:

1. **Large scales (low $L$): the diffusion wins big** ($\rho\!\approx\!0.95$ vs the
   EB QE's $\sim\!0.71$). Here it pays to use the *full*, non-Gaussian field-level
   likelihood — lensing maps $\phi$ into $E\!\to\!B$ non-linearly, and the
   *leading-order* quadratic estimator discards that higher-order information.
2. **Small scales (high $L$): the EB QE overtakes it** ($\sim\!0.66$ vs $\sim\!0.54$).
   There the analytic estimator is near-optimal, while the learned U-Net's finite
   capacity/receptive field and $256^2$ resolution limit how faithfully it recovers
   the highest-$L$ $\phi$ modes. A learned field-level method is **not** automatically
   better everywhere — even on identical data.
3. **The real win is the *calibrated posterior*.** The per-mode pull below has
   std $\approx 1$ — the diffusion's error bars are honest, which the QE only gets via
   its analytic $N_0$. *That*, not a uniform $\rho$ advantage, is the payoff.

Caveats kept in view: the diffusion is trained on the **exact** generative model
(one cosmology, exact noise, periodic flat sky, no foregrounds/mask), and its
posterior *mean* is implicitly Wiener-filtered while the QEs are raw — we compare via
the normalization-independent $\rho_L$ to keep that fair.

**Try it:** re-run the test sims with a different cosmology (scale `pp2d` by 1.3, or
raise the noise) *without* retraining — the QEs are largely unaffected (they assume
no learned $\phi$ prior), while the diffusion, which baked in one prior, degrades.
That is the price of a learned method.

### Validate the *posterior*, not just the mean (coverage)
The diffusion's whole advantage over the QE is that it returns a full posterior —
so we must check its *uncertainty* is honest, not just its mean. Per Fourier mode,
the pull $(\phi_{\rm true}-\bar\phi)/\sigma_\phi$ across the posterior samples
should be $\mathcal{N}(0,1)$ if the posterior is calibrated. This is the
field-level analogue of Act 2's SBC, and it is the check that most directly backs
the "trustworthy posterior" claim.

> **Finite-sample caveat:** each mode's mean *and* spread are estimated from only
> `n_post` samples, so even a perfectly calibrated posterior gives a pull std of
> $\sqrt{(1+1/n)(n-1)/(n-3)}>1$, not exactly 1. We print that floor next to the
> measured value — "calibrated" means *matching the floor*, not beating it.

In [ ]:
# pull = (phi_true - sample_mean) / sample_std per Fourier mode, over the n_post draws.
# (Real + imag parts: a real field's FFT is Hermitian, so k and -k are conjugate --
# harmless double-counting that only changes the effective mode count, not mean/std.)
band_cov = (fs.ell >= 40) & (fs.ell <= 400)
pulls = []
for j in range(n_test):
    fk = np.fft.fft2(phi_samples[:, j], axes=(1, 2))      # (n_post, H, W) complex
    ft = np.fft.fft2(test[j]["phi"])
    for tr, mu, sd in [(ft.real, fk.real.mean(0), fk.real.std(0, ddof=1)),
                       (ft.imag, fk.imag.mean(0), fk.imag.std(0, ddof=1))]:
        with np.errstate(divide="ignore", invalid="ignore"):
            p = ((tr - mu) / sd)[band_cov]
        pulls.append(p[np.isfinite(p)])
pulls = np.concatenate(pulls)
# Finite-sample floor: mean AND std are estimated from only n_post draws, so a
# perfectly calibrated posterior gives pull std = sqrt((1+1/n)(n-1)/(n-3)) > 1, not 1.
pull_std = pulls.std()
floor = np.sqrt((1 + 1 / n_post) * (n_post - 1) / (n_post - 3))
print(f"posterior pull: mean {pulls.mean():+.2f}, std {pull_std:.2f}  "
      f"(calibrated -> mean ~0, std ~{floor:.2f} at n_post={n_post}; much larger = overconfident)")

xx = np.linspace(-4, 4, 200)
plt.figure(figsize=(5.5, 3.5))
plt.hist(np.clip(pulls, -4, 4), bins=60, density=True, alpha=0.6,
         label=f"diffusion pulls (std={pull_std:.2f}; calibrated≈{floor:.2f})")
plt.plot(xx, np.exp(-xx ** 2 / 2) / np.sqrt(2 * np.pi), "k-", label=r"$\mathcal{N}(0,1)$")
plt.xlabel(r"per-mode pull $(\phi_{\rm true}-\bar\phi)/\sigma_\phi$")
plt.legend(); plt.title("Is the diffusion $\\phi$ posterior calibrated?")

### Delensing payoff: cleaner B-modes
Why reconstruct $\phi$ at all? Subtracting the lensing-induced $B$-modes uncovers
any *primordial* $B$ signal — the goal of tensor-to-scalar-ratio ($r$) searches.
With a $\phi$ tracer of correlation $\rho_L$ with the truth, optimal delensing
removes a fraction $\approx\rho_L^2$ of the lensing-$B$ power, leaving
$(1-\rho_L^2)$. So a better $\hat\phi$ delenses more — the diffusion's higher
$\rho_L$ turns directly into a cleaner $B$-mode sky.

(Full *map-level* template delensing also needs $\phi$ on small scales, where a
temperature-only reconstruction is weakest; in practice one uses the $EB$
estimator or a deep $\phi$ map. Here we quote the efficiency from $\rho_L$.)

In [ ]:
plt.figure()
plt.plot(Lb, 1 - rho_qe ** 2, "o-", color="C0", label="QE: temperature (TT)")
plt.plot(Lb, 1 - rho_eb ** 2, "^-", color="C2", label="QE: polarization (EB)")
plt.plot(Lb, 1 - rho_diff ** 2, "s-", color="C3", label="diffusion (E,B)")
plt.axhline(1.0, color="k", lw=0.8, ls=":", label="no delensing")
plt.ylim(0, 1.05); plt.xlabel("L")
plt.ylabel(r"residual lensing-$B$ fraction  $1-\rho_L^2$")
plt.legend(); plt.title("Delensing efficiency (lower = more lensing B removed)")
print("residual lensing-B fraction, diffusion:", np.round(1 - rho_diff ** 2, 2))

**Takeaways.**
- The QE is a well-understood, leading-order baseline; the diffusion model is a
  learned, *probabilistic* alternative that returns a full posterior over $\phi$.
- We validated it on **both** axes: its **mean** against the known input $\phi$
  (cross-correlation), *and* its **uncertainty** with a per-mode coverage test —
  the point estimate winning means little if the error bars are wrong.
- The comparison is **fair** (diffusion and EB QE use the *same* $E,B$ data), and the
  honest result is a **crossover**: the field-level diffusion wins at large scales
  (it captures higher-order lensing info the leading-order QE discards), while the EB
  QE wins at small scales. The robust, transferable win is that — unlike the QE — the
  diffusion returns a **calibrated** posterior (per-mode pull std $\approx 1$).
- Better $\hat\phi$ → more lensing $B$ removed → a cleaner path to primordial
  $B$-modes, tying delensing back to the inference goals of Acts 1–2.